# 04 · Trajectory figures and architecture diagram
Table and figure numbers refer to manuscript draft v5, in which all tables are in Section 5 (there is no appendix). All figures are saved to `../figures/` at 600 dpi.

| Figure file | Illustrates (draft v5) |
|---|---|
| `Fig01_architecture.png` | Figure 1: HRL-H architecture |
| `Fig03_traj_HRLH_2x2.png` | Figure 3: HRL-H trajectories, four environments (environment-figure design) |
| `Fig05_traj_all_open.png`, `Fig06_traj_all_layered.png`, `Fig07_traj_all_open_nfz.png`, `Fig08_traj_all_layered_nfz.png` | Figures 5-8: all eight methods per environment |

The episodes were recorded with the project code (`support_scripts/record_episodes.py`) and are stored in `../data/episodes_seed0.pkl`; they match the project's `trajectories_<env>.json` exactly. Trail colour = final SOC (green = full, red = depleted); triangles = numbered UAV start positions; circles = end positions; stars = tasks.

In [ ]:
import sys
sys.path.insert(0, "..")          # common.py lives in the package root
from common import *
%matplotlib inline
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

EP = pickle.load(open(DATA / "episodes_seed0.pkl", "rb"))
ENVTITLE = {"open": ("Open volume", "volume", "no"), "layered": ("Layered altitude bands", "layered", "no"),
            "open_nfz": ("Open volume + no-fly zones", "volume", "yes"), "layered_nfz": ("Layered + no-fly zones", "layered", "yes")}
CM = plt.cm.RdYlGn

In [ ]:
# ================= environment-design trajectory figures (match Figure 2 style) =================
from matplotlib.lines import Line2D
CM=plt.cm.RdYlGn
ENVTITLE={'open':('Open volume','volume','no'),'layered':('Layered altitude bands','layered','no'),
          'open_nfz':('Open volume + no-fly zones','volume','yes'),'layered_nfz':('Layered + no-fly zones','layered','yes')}
def draw_env2(ax,e,name,title,fs=7):
    d=EP[e]; p=d['pol'][name]; tr=p['traj']; soc=np.clip(p['soc'][-1] if np.ndim(p['soc'])==2 else p['soc'],0,1)
    for o in d['obstacles']:
        x0,y0,z0,x1,y1,z1=o
        v=[[x0,y0,z0],[x1,y0,z0],[x1,y1,z0],[x0,y1,z0],[x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1]]
        f=[[v[i] for i in idx] for idx in [[0,1,2,3],[4,5,6,7],[0,1,5,4],[2,3,7,6],[1,2,6,5],[0,3,7,4]]]
        ax.add_collection3d(Poly3DCollection(f,facecolors='#d9534f',alpha=0.30,edgecolors='#b04a48',linewidths=0.5))
    for i in range(tr.shape[1]):
        c=CM(soc[i])
        ax.plot(tr[:,i,0],tr[:,i,1],tr[:,i,2],color=c,lw=1.1,alpha=0.95)
        ax.scatter(*tr[-1,i],color=c,s=14,marker='o',edgecolor='k',linewidth=0.4,zorder=6,depthshade=False)
        ax.scatter(*tr[0,i],color='#1a7f4b',s=34,marker='^',edgecolor='k',linewidth=0.5,zorder=7,depthshade=False)
        ax.text(tr[0,i,0]+8,tr[0,i,1],tr[0,i,2]+1.5,str(i),fontsize=fs-1.2)
    tp=d['task_pos']; c=p['completed']
    ax.scatter(tp[c,0],tp[c,1],tp[c,2],marker='*',s=62,color='#f5a623',edgecolor='k',linewidth=0.6,zorder=5,depthshade=False)
    if (~c).any(): ax.scatter(tp[~c,0],tp[~c,1],tp[~c,2],marker='X',s=40,color='#d62728',edgecolor='k',linewidth=0.5,zorder=5,depthshade=False)
    ax.scatter(*d['base'],marker='s',s=48,color='k',zorder=8,depthshade=False)
    ax.set_xlim(0,400); ax.set_ylim(0,400); ax.set_zlim(0,60); ax.set_box_aspect((1,1,0.62)); ax.view_init(27,-58)
    ax.set_xlabel('x (m)',fontsize=fs,labelpad=-5); ax.set_ylabel('y (m)',fontsize=fs,labelpad=-5); ax.set_zlabel('altitude z (m)',fontsize=fs,labelpad=-3)
    ax.tick_params(labelsize=fs-1.5,pad=-3); ax.set_xticks([0,100,200,300,400]); ax.set_yticks([0,100,200,300,400]); ax.set_zticks([0,20,40,60])
    for a in (ax.xaxis,ax.yaxis,ax.zaxis): a.pane.set_facecolor((0.95,0.95,0.95,1)); a.pane.set_edgecolor('#cccccc')
    ax.grid(True,linewidth=0.3)
    ax.set_title(title,fontsize=fs+0.6,pad=1,linespacing=1.25)
def legend_env(fig,nfz=True,y=0.0):
    h=[Line2D([],[],marker='s',ls='',color='k',ms=5,label='Base / depot'),
       Line2D([],[],marker='^',ls='',color='#1a7f4b',mec='k',mew=0.4,ms=6,label='UAV start'),
       Line2D([],[],marker='o',ls='',color=CM(1.0),mec='k',mew=0.4,ms=4.5,label='UAV end (trail colour = final SOC)'),
       Line2D([],[],marker='*',ls='',color='#f5a623',mec='k',mew=0.5,ms=9,label='Task (completed)'),
       Line2D([],[],marker='X',ls='',color='#d62728',mec='k',mew=0.4,ms=6,label='Task (not completed)')]
    if nfz: h.append(Rectangle((0,0),1,1,fc='#d9534f',alpha=0.35,ec='#b04a48',label='No-fly zone (cuboid)'))
    fig.legend(handles=h,loc='lower center',ncol=3 if len(h)>5 else 3,fontsize=6.6,frameon=False,bbox_to_anchor=(0.5,y),columnspacing=1.4,handletextpad=0.3)
def traj_hrlh2():
    fig=plt.figure(figsize=(6.6,7.0))
    for k,e in enumerate(ENVS):
        nm,sc,nf=ENVTITLE[e]; p=EP[e]['pol']['hrlh']
        ax=fig.add_subplot(2,2,k+1,projection='3d')
        draw_env2(ax,e,'hrlh',f'{nm}\ntasks=8, UAVs=8, scenario={sc}, NFZ={nf}\nHRL-H: {int(p["completed"].sum())}/8 tasks, {p["steps"]} steps',fs=7.2)
    fig.subplots_adjust(left=0.0,right=0.97,top=0.94,bottom=0.08,wspace=0.02,hspace=0.17); legend_env(fig,True,0.0)
    save(fig, 'Fig03_traj_HRLH_2x2')
def traj_all2(e):
    order=['random','greedy','cbba','mappo','maddpg','qmix','dmpc','hrlh']; nm,sc,nf=ENVTITLE[e]
    fig=plt.figure(figsize=(6.6,9.6))
    fig.suptitle(f'{nm} (tasks=8, UAVs=8, scenario={sc}, NFZ={nf})',fontsize=8.6,y=0.995)
    for k,m in enumerate(order):
        p=EP[e]['pol'][m]; ax=fig.add_subplot(4,2,k+1,projection='3d')
        draw_env2(ax,e,m,f'({"abcdefgh"[k]}) {MN[m]}\n{int(p["completed"].sum())}/8 tasks, {p["steps"]} steps',fs=7.0)
    fig.subplots_adjust(left=0.0,right=0.97,top=0.955,bottom=0.05,wspace=0.02,hspace=0.16); legend_env(fig,'nfz' in e,0.0)
    save(fig, {'open': 'Fig05_traj_all_open', 'layered': 'Fig06_traj_all_layered', 'open_nfz': 'Fig07_traj_all_open_nfz', 'layered_nfz': 'Fig08_traj_all_layered_nfz'}[e])


In [ ]:
traj_hrlh2()                       # Fig_traj_HRLH_2x2
for e in ENVS:
    traj_all2(e)                  # Fig_traj_all_<env>

## Architecture diagram

In [ ]:
def arch():
    fig,ax=plt.subplots(figsize=(7.0,4.9)); ax.set_xlim(0,100); ax.set_ylim(0,70); ax.axis('off')
    def box(x,y,w,h,txt,fc,ec,fs=7.2,bold=False,lw=0.9,tc='k'):
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.25,rounding_size=1.2',fc=fc,ec=ec,lw=lw))
        ax.text(x+w/2,y+h/2,txt,ha='center',va='center',fontsize=fs,fontweight='bold' if bold else 'normal',linespacing=1.25,color=tc)
    def arr(p,q,c='#333',ls='-'):
        ax.add_patch(FancyArrowPatch(p,q,arrowstyle='-|>',mutation_scale=9,color=c,lw=1.0,ls=ls))
    ax.add_patch(FancyBboxPatch((1,43),69,25,boxstyle='round,pad=0.3,rounding_size=1.5',fc='#eef3fb',ec='#2a5db0',lw=1.2))
    ax.text(3,65.3,'HIGH LEVEL (learned) - what to do',fontsize=8.6,fontweight='bold',color='#1d4587',va='center')
    box(3,46,20,15,'Observation $o_i^t$\nown $(x,y,z,\\mathrm{SOC})$\n$K{=}5$ neighbour beliefs\ntask $(x,y,z,\\mathrm{active})$','white','#2a5db0',6.8)
    box(26.5,46,17,15,'MAPPO actor\n(attention over\nneighbours)\nor QMIX Q-net','#2a5db0','#2a5db0',7,True,tc='white')
    box(47.5,46,20,15,'Task preference\n$\\rho_{ij}\\in[0,1]$\nmin-max normalised\nevery step','white','#2a5db0',6.8)
    arr((23.5,53.5),(26.2,53.5),'#2a5db0'); arr((43.8,53.5),(47.2,53.5),'#2a5db0')
    ax.add_patch(FancyBboxPatch((1,3),69,35,boxstyle='round,pad=0.3,rounding_size=1.5',fc='#eef7ee',ec='#2b7a2b',lw=1.2))
    ax.text(3,35.3,'LOW LEVEL (heuristic executor) - how to get there',fontsize=8.6,fontweight='bold',color='#1d5e1d',va='center')
    box(2.6,6,15.6,25.5,'1  Battery safety\n\nSOC < SOC$_{min}$:\nreturn to base\n\nprefer tasks with a\nsafe round-trip\n(known dynamics)','white','#2b7a2b',6.3)
    box(19.8,6,15.6,25.5,'2  Commit on approach\n\nif assigned task is\nactive and\n$\\|p_i-q_j\\|<d_c=12$ m,\nkeep it until\ncompleted','white','#2b7a2b',6.3)
    box(37,6,16.6,25.5,'3  Global auction\n\n$s_{ij}=-w_e\\hat E_{ij}-w_t\\hat T_{ij}$\n$+\\,w_\\rho\\rho_{ij}+w_d u_j$\ngreedy best pair;\ncapacity $R_j$ per task\n(deconfliction)','white','#2b7a2b',6.3)
    box(55.2,6,13.6,25.5,'4  Navigation\n\nfly at $v_{max}$\nto $q_j$ or base\n\nslide along\nno-fly faces;\nhold if idle','white','#2b7a2b',6.3)
    for x in (18.5,35.7,53.9): arr((x-0.1,18.7),(x+1.1,18.7),'#2b7a2b')
    arr((40,42.7),(40,38.6),'#2a5db0'); ax.text(41.6,40.7,'$\\rho_{ij}$ enters as a bounded tie-break ($w_\\rho=0.03$)',fontsize=6.5,color='#2a5db0',va='center')
    ax.add_patch(FancyBboxPatch((80,12),19,44,boxstyle='round,pad=0.3,rounding_size=1.5',fc='#f6f0fa',ec='#6a3d9a',lw=1.2))
    ax.text(89.5,52.5,'ENVIRONMENT',fontsize=8.2,fontweight='bold',color='#4a2a72',ha='center',va='center')
    ax.text(82,31,'Tasks (3D)\nNo-fly zones\nBase / depot\nBattery dynamics\nRange-limited,\nlossy comm.\n(20% base loss)\nUAV dropout',fontsize=6.8,va='center',ha='left',linespacing=1.4)
    arr((69.2,18.7),(79.6,18.7),'#2b7a2b'); ax.text(74.4,21.4,'actions\n(every step)',fontsize=6.2,ha='center',va='bottom',color='#2b7a2b')
    arr((79.6,55),(70.6,56),'#555','--'); ax.text(75.1,58.6,'observations',fontsize=6.2,ha='center',color='#555')
    ax.plot([89.5,89.5],[11.6,1.2],color='#555',lw=1.0,ls='--'); ax.plot([89.5,10],[1.2,1.2],color='#555',lw=1.0,ls='--')
    arr((10,1.2),(10,2.6),'#555','--')
    ax.text(50,0.2,'shared fleet / task state to the allocator',fontsize=6.4,ha='center',va='top',color='#555')
    save(fig, 'Fig01_architecture')

arch()